# Trial Pipeline Health Summary

Reads the `clinical_trials.trial_registry` bronze table (see
`data-generators/clinical_trials_registry_ingest.ipynb`) and summarizes
trial portfolio health by phase and status — a CRO/biotech R&D operations
view: how many trials are in each phase, how enrollment scales by phase,
and where trials are terminating.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "clinical_trials"
SOURCE_TABLE = "trial_registry"
TARGET_TABLE = "trial_pipeline_health_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: pipeline health by phase

Aggregates by trial phase: trial count, average enrollment target,
terminated-trial count, and termination rate.

In [ ]:
summary_df = (
    df
    .groupBy("phase")
    .agg(
        F.count("*").alias("trial_count"),
        F.round(F.avg("enrollment_count"), 1).alias("avg_enrollment"),
        F.sum(F.when(F.col("overall_status") == "TERMINATED", 1).otherwise(0)).alias("terminated_count"),
        F.sum(F.when(F.col("overall_status") == "COMPLETED", 1).otherwise(0)).alias("completed_count"),
        F.sum(F.when(F.col("overall_status").isin("RECRUITING", "NOT_YET_RECRUITING"), 1).otherwise(0)).alias("active_recruiting_count"),
    )
    .withColumn("termination_rate_pct", F.round(F.col("terminated_count") * 100.0 / F.col("trial_count"), 1))
    .orderBy(F.desc("trial_count"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_phases FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY trial_count DESC").show()